# NoLA on EuroSAT 


## Settings
Keep `recipe = "teaching"` for the 160-image exercise. Choose `recipe = "release"` for the full training and validation splits and the authors’ released hyperparameters, adapted for a small GPU. Restart the kernel before changing recipes so CLIP starts from its original weights. The full recipe is much longer; lower `prompt_microbatch` if GPU memory is tight (this does not change the effective batch). Use the optional test cell only after choosing the recipe on validation.

In [ ]:
recipe = "release"  # Use "release" for the full run.
clip_backbone = "ViT-B/32"
clip_score_batch, inference_batch = 8, 8
head_lr = 0.001
visual_prompts = 16
prompt_epochs = 50
prompt_label_smoothing = 0.1

if recipe == "release":
    seed = 7777
    train_size, eval_size, eval_split = None, None, "val"  # None = whole split.
    anchor_cap = None  # Compute k using the released rule.
    head_epochs, head_batch = 50, 32
    prompt_batch, prompt_microbatch, prompt_lr = 512, 4, 0.004
elif recipe == "teaching":
    seed = 42
    train_size, eval_size, eval_split = 160, 80, "test"
    anchor_cap = 4
    head_epochs, head_batch = 3, 16
    prompt_batch, prompt_microbatch, prompt_lr = 8, 8, 0.001
else:
    raise ValueError("Choose 'teaching' or 'release'.")

## 1. Open EuroSAT
`split` contains lists of `[image path, class ID, class name]`. `names[id]` gives the name of a classifier column. `random.sample` picks a small repeatable subset without looking at class labels. The evaluation subset is for scoring only; the release recipe uses validation. `load_images` uses Pillow for RGB decoding and the supplied library transform for pixel preparation. It stacks tensors into `[batch, 3, 224, 224]`; we reuse it for each model instead of writing a custom Dataset. Run this cell and look at the printed path and image.

In [ ]:
# Path builds filesystem paths with /. JSON reads the split, random samples it, and sys configures imports.
from pathlib import Path
import json, random, sys
from math import ceil

# nn provides model layers; F provides tensor operations and losses. Pillow opens images; display previews them.
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from IPython.display import display

# Locate this walkthrough and the pinned NoLA checkout.
root = Path.home() / "nola-walkthrough"
upstream = root / "upstream"
# Keep Python from writing bytecode caches into the upstream checkout.
sys.dont_write_bytecode = True
# Search these directories first so the following imports pick up NoLA's own CLIP and helpers.
sys.path[:0] = [str(upstream), str(root / ".deps")]
from clip import clip
from utils.data_utils import ds_specific_templates
from utils.model_utils import te_transform, tr_transforms, top_k_indices_per_class2

# Seed Python for augmentation and PyTorch for shuffling and weight initialization.
random.seed(seed)
torch.manual_seed(seed)
# Point to the RGB JPEGs, then read the published split JSON from the parent directory.
image_dir = root / "data/eurosat/EuroSAT_RGB"
split = json.loads((image_dir.parent / "split_zhou_EuroSAT.json").read_text())
# Read the class names by integer ID; sorting the IDs fixes classifier column order.
name_by_id = {label: name for _, label, name in split["train"]}
names = [name_by_id[i] for i in sorted(name_by_id)]
# Use local seeded samplers to pick distinct rows from the two disjoint splits.
train = split["train"] if train_size is None else random.Random(seed).sample(split["train"], train_size)
eval_rows = split[eval_split] if eval_size is None else random.Random(seed).sample(split[eval_split], eval_size)
# The release floors 20% of the average images per class and applies a minimum of 16.
if anchor_cap is None:
    anchor_cap = max(16, int(0.2 * len(train) / len(names)))
print(f"{recipe}: {len(train)} train, {len(eval_rows)} {eval_split}, anchor cap {anchor_cap}")
print(f"CLIP updates/epoch: {ceil(len(train) / prompt_batch)}; effective batch {prompt_batch}, microbatch {prompt_microbatch}")
# Keep held-out integer labels for evaluation only.
y_eval = torch.tensor([label for _, label, _ in eval_rows])


# Read a batch of split rows with whichever transform CLIP or DINO needs.
def load_images(rows, transform):
    batch = []
    # Use only the relative path from each row, not its label or name.
    for path, _, _ in rows:
        with Image.open(image_dir / path) as image:
            # Convert to RGB and transform the image while the file is open.
            batch.append(transform(image.convert("RGB")))
    # Stack individual [3, 224, 224] tensors into a batch and move it to the GPU.
    return torch.stack(batch).cuda()


# Show the split sizes, class order, and one row. The final resize is only for display.
print(len(train), "adaptation images;", len(eval_rows), "held-out images")
print(dict(enumerate(names)))
print(train[0])
with Image.open(image_dir / train[0][0]) as image:
    display(image.resize((192, 192)))

## 2. Build the class description embeddings (CDE)
`clip.load()` returns the pretrained model, embedding width, and image transform. `eval()` selects inference behavior; `no_grad()` disables gradient recording for the text pass. For each class, the JSON descriptions and formatted templates are separate strings. `F.normalize(..., dim=1)` makes each prompt vector unit length; `.mean(dim=0)` averages the prompts. The final normalization sets the mean's length back to 1. The stack puts the vectors into class **columns**, giving `[512, 10]`. This is the calculation in `gen_emb()` expressed with PyTorch library calls.

In [ ]:
# Load local CLIP weights. Also get its embedding width and matching image transform.
model, width, preprocess = clip.load(clip_backbone, device="cuda", download_root=str(root / "weights"))
# Select evaluation behavior; no_grad below separately disables gradient tracking.
model.eval()
# Load the saved LLM descriptions and NoLA's separate CLIP phrase templates.
descriptions = json.loads((upstream / "descriptions/generic/EuroSAT.json").read_text())
templates = ds_specific_templates["EuroSAT"]

# Collect one finished embedding per class; no text encoder training here.
weights = []
with torch.no_grad():
    for name in names:
        # Join two lists of strings. The templates are not attached to each description.
        texts = descriptions[name] + [t.format(name) for t in templates]
        # Make [texts, 77] integer token IDs and move them to CLIP's device.
        tokens = clip.tokenize(texts).cuda()
        # Encode to [texts, 512], then normalize each row across its features (dim=1).
        vectors = F.normalize(model.encode_text(tokens), dim=1)
        # Mean over prompt rows (dim=0); normalize the resulting 512-feature vector again.
        weights.append(F.normalize(vectors.mean(dim=0), dim=0))
# Stack class vectors as columns: [512 features, 10 classes].
cde = torch.stack(weights, dim=1)
print("CDE weights [features, classes]:", tuple(cde.shape))

## 3. Score the images with CLIP
Study `forward_with_text_desc()` in `NoLA.py`. `encode_image()` gives `[batch, 512]` features. The dot product of the normalized image features and CDE columns gives one cosine similarity per class. CLIP multiplies by its learned scale; softmax compares classes within each image. `topk(3)` displays three candidate classes for the first image. The held-out count is our unprompted CDE baseline, not a training signal.

In [ ]:
# No training: this function only scores images.
@torch.no_grad()
# Return one row of class scores per image, in the same order as the input rows.
def clip_scores(rows):
    scores = []
    # Slice into batches of 16 to keep GPU use small.
    for i in range(0, len(rows), clip_score_batch):
        # Apply the pretrained CLIP image transform, not the display resize.
        images = load_images(rows[i:i+clip_score_batch], preprocess)
        # Encode images and make every [512] feature row unit length.
        features = F.normalize(model.encode_image(images), dim=1)
        # Multiply [batch, 512] by [512, 10] for cosine similarities; apply CLIP's learned scale.
        logits = model.logit_scale.exp() * (features @ cde)
        # Softmax over classes (dim=1), then store the scores on the CPU.
        scores.append(logits.softmax(dim=1).cpu())
    # Join the batches along the image axis (default dim=0).
    return torch.cat(scores)


# Keep adaptation scores for the selector; score eval_rows images separately for the baseline.
probs = clip_scores(train)
# argmax picks a class per image; count matches to the held-out labels.
clip_correct = (clip_scores(eval_rows).argmax(dim=1) == y_eval).sum().item()
print("Adaptation score matrix:", tuple(probs.shape))
print("Unprompted CDE held-out correct:", clip_correct, "/", len(eval_rows))
# Get the first image's three best scores and map their column indices to names.
values, ids = probs[0].topk(3)
print("First image's top three:", [(names[i], round(v, 3)) for i, v in zip(ids.tolist(), values.tolist())])

## 4. Pick anchors using the original selector
`top_k_indices_per_class2()` is from [`utils/model_utils.py`](upstream/utils/model_utils.py). It ranks images within each predicted class and keeps up to `k=4`. Only if a class has no first-choice candidates does it try the second, then third choice; it also removes duplicate image IDs. The labels tensor is required by the released helper to enumerate present classes and print diagnostics. It does **not** supply the training target for an image. The targets are `pseudo`, the selector's assignments. Our count compares the returned pseudo-labels with withheld audit labels; this is useful because the helper's printed aggregate may still use the original top-1 labels after fallback. Inspect one row: image path, assigned class, audit class.

In [ ]:
# These true adaptation labels are for the released helper's class list and audits, not training targets.
audit = torch.tensor([label for _, label, _ in train])
# Run the upstream selector. Keep image indices and assigned classes; discard returned confidence.
anchor_ids, _, pseudo = top_k_indices_per_class2(
    # Give the helper scores, row indices, and audit labels. k is a cap per class, not a guarantee.
    {"total_emb": probs, "idxs": torch.arange(len(train)), "labels": audit}, k=anchor_cap
)
# Map returned row indices back to the original adaptation rows.
anchors = [train[i] for i in anchor_ids.tolist()]
# Compare the final assigned labels (including fallback) with their true labels.
print("Correct returned anchor labels:", (audit[anchor_ids] == pseudo).sum().item(), "/", len(anchors))
# Show one selected file, its assigned class, and its true class side by side.
print("First anchor:", anchors[0][0], "| pseudo:", names[pseudo[0].item()], "| audit:", anchors[0][2])

## 5. Extract frozen DINO features
`torch.hub.load` constructs the published DINO ViT-B/16 from the local checkout. We then load the downloaded weights. The `try/finally` is only there because both repositories call a module `utils`; it restores NoLA's module after the DINO import. GPU memory is limited, so CLIP waits on the CPU during this stage. `te_transform` is the original NoLA weak/test view. The output `[images, 768]` describes how DINO sees each image. There's no training here, and these features contain no class scores yet.

In [ ]:
# Move CLIP off the GPU during DINO extraction. empty_cache releases unused allocator blocks.
model.cpu()
torch.cuda.empty_cache()
# Both checkouts have a module called utils. Temporarily allow the local DINO import to find its own.
nola_utils = sys.modules.pop("utils")
try:
    # Construct the local DINO ViT-B/16 without asking Torch Hub to download weights.
    dino = torch.hub.load(str(root / "dino-source"), "dino_vitb16", source="local", pretrained=False)
finally:
    # Restore NoLA's utils even if DINO's import fails.
    sys.modules["utils"] = nola_utils
# Read the downloaded tensor weights on CPU, copy them into DINO, then drop the extra copy.
state = torch.load(root / "weights/dino_vitbase16_pretrain.pth", map_location="cpu", weights_only=True)
dino.load_state_dict(state)
del state
# Use inference behavior, freeze DINO's parameters, then move it to the GPU.
dino.eval().requires_grad_(False).cuda()


# Feature extraction does not need a backward pass.
@torch.no_grad()
# Apply NoLA's weak/eval_rows transform in groups of 8 and concatenate CPU feature rows.
def dino_features(rows):
    return torch.cat([dino(load_images(rows[i:i+inference_batch], te_transform)).cpu()
                      for i in range(0, len(rows), inference_batch)])


# Keep the original train and eval_rows row orders; each image yields 768 DINO features.
train_features = dino_features(train)
eval_features = dino_features(eval_rows)
# Move DINO back to CPU so prompted CLIP can use GPU memory later.
dino.cpu()
torch.cuda.empty_cache()
print("Frozen DINO features:", tuple(train_features.shape))

## 6. Fit the DINO classifier head
This `nn.Sequential` is the architecture of upstream `AdapterMLP`: `768 → 256 → classes`, with a ReLU in between. Indexing `train_features` by `anchor_ids` pairs each DINO vector with the pseudo-label from the **same image**. Adam and cross-entropy update only the head. After training the head, `argmax` gives a hard pseudo-label from the DINO head for *every* adaptation image, not just anchors. The next stage uses this label to tune CLIP. Compare held-out accuracy to CDE, not just the last training loss.

In [ ]:
# Build NoLA's 768 → 256 → class-count head using standard Linear and ReLU layers.
dino_head = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, len(names)))
# Adam sees only the head parameters, not the frozen DINO encoder.
optimizer = torch.optim.Adam(dino_head.parameters(), lr=head_lr)
# Run the configured teaching epochs; randperm shuffles anchor positions and split makes minibatches.
for epoch in range(head_epochs):
    for ids in torch.randperm(len(anchors)).split(head_batch):
        # Convert anchor positions to adaptation row IDs, then score those cached DINO features.
        logits = dino_head(train_features[anchor_ids[ids]])
        # Compare against the matching CLIP pseudo-labels, not the true audit labels.
        loss = F.cross_entropy(logits, pseudo[ids])
        # Clear the previous batch's gradients, compute new ones, and take an Adam step.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # This is the last batch loss, not the epoch mean or held-out accuracy.
    print("DINO head epoch", epoch + 1, "last batch loss:", round(loss.item(), 3))

# Inference only: assign one hard class ID to every adaptation image and evaluate on eval_rows.
with torch.no_grad():
    dino_pseudo_labels = dino_head(train_features).argmax(dim=1)
    dino_head_correct = (dino_head(eval_features).argmax(dim=1) == y_eval).sum().item()
print("DINO pseudo-labels:", tuple(dino_pseudo_labels.shape))
print("DINO head held-out correct:", dino_head_correct, "/", len(eval_rows))

## 7. Tune CLIP with visual prompts
`embeddings_patch()` and `forward_after_patch_embeddings()` are the original CLIP fork methods. The only custom forward logic is the NoLA operation between them: insert 16 learned vectors by default **after CLS and before the image patches**. `.expand()` gives each image the same learned prompt; the vectors are initialized using NoLA's uniform bound. The classifier starts from `cde.T` (PyTorch Linear stores one class per row). The optimizer updates the prompt, classifier and visual normalization parameters, not the rest of CLIP.

The upstream strong augmentation is `tr_transforms(image)[1]`; the hard targets from the DINO head were computed from the deterministic weak view in the previous cells. `F.cross_entropy(..., label_smoothing=0.1)` uses the same smoothing rule as NoLA without importing another loss package. In release mode, the 512-image effective batch is processed in small microbatches; gradients accumulate before AdamW updates. Each sample carries the same weight even in the final partial batch. The best prompted CLIP weights are saved by validation accuracy. The teaching preset retains the 80-image exploratory test comparison. The initial prompted model is not plain CLIP: inserting random visual prompts changes its forward pass.

**Saved teaching run before adding the preset:** 49/80 after 50 epochs with 16 prompts. The notebook backup retains the original output. This release-inspired path keeps a fixed learning rate and uses gradient accumulation, so it is not a bit-for-bit paper reproduction.

In [ ]:
# Return CLIP to the GPU in float32. eval() does not prevent backpropagation.
model.float().cuda().eval()
# Freeze the backbone except visual normalization parameters, following the released name filter.
for name, param in model.named_parameters():
    param.requires_grad_("visual" in name and ("ln" in name or "bn" in name))

# Use CLIP's visual token width and the same initialization bound as upstream NoLA.
hidden = model.visual.class_embedding.numel()
bound = (6 / (3 * 16 * 16 + hidden)) ** 0.5
# Create trainable visual tokens with shape [1, visual_prompts, hidden].
prompt = nn.Parameter(torch.empty(1, visual_prompts, hidden, device="cuda").uniform_(-bound, bound))
# Use a bias-free Linear head with one row per class; initialize it with the transposed CDE matrix.
classifier = nn.Linear(width, len(names), bias=False).cuda()
with torch.no_grad():
    classifier.weight.copy_(cde.T)


# Keep this small forward function visible so the visual prompt insertion is easy to inspect.
def prompted_clip_logits(images):
    # Make the CLS and patch tokens using the original CLIP vision encoder method.
    tokens = model.visual.embeddings_patch(images)
    # Expand the shared prompt across the batch and insert it between CLS and the image patches.
    tokens = torch.cat([tokens[:, :1], prompt.expand(len(images), -1, -1), tokens[:, 1:]], dim=1)
    # Run the remaining visual transformer, then map the image embedding to class logits.
    return classifier(model.visual.forward_after_patch_embeddings(tokens))


# Disable gradient tracking for held-out evaluation only.
@torch.no_grad()
# Predict a class per eval_rows image in batches and count matches with the true eval_rows labels.
def prompted_clip_correct(rows=eval_rows, labels=y_eval):
    pred = [prompted_clip_logits(load_images(rows[i:i+inference_batch], te_transform)).argmax(dim=1).cpu()
            for i in range(0, len(rows), inference_batch)]
    return (torch.cat(pred) == labels).sum().item()


# The upstream transform returns two views; index 1 selects the strongly augmented view.
def strong(image):
    return tr_transforms(image)[1]


# List only the CLIP parameters that the optimizer may update.
trainable = [(name, param) for name, param in model.named_parameters() if param.requires_grad]
# AdamW updates the prompts, classifier, and selected visual parameters. Biases get no weight decay.
optimizer = torch.optim.AdamW([
    {"params": [prompt, *classifier.parameters(), *(p for n, p in trainable if "bias" not in n)]},
    {"params": [p for n, p in trainable if "bias" in n], "weight_decay": 0.0},
], lr=prompt_lr)

# Evaluate before training, then track the best validation weights for the release recipe.
before = prompted_clip_correct()
best_val, best_epoch = -1, None
adapted_params = [prompt, classifier.weight, *(p for _, p in trainable)]
checkpoint_path = root / "eurosat-release-best.pt"

import time
start = last_report = time.monotonic()
updates = 0
planned = prompt_epochs * ceil(len(train) / prompt_batch)
for epoch in range(1, prompt_epochs + 1):
    for ids in torch.randperm(len(train)).split(prompt_batch):
        # One optimizer update per effective batch, not per microbatch.
        optimizer.zero_grad()
        for micro_ids in ids.split(prompt_microbatch):
            images = load_images([train[i] for i in micro_ids.tolist()], strong)
            # Add sample losses, dividing by the actual effective batch length (including the tail).
            loss = F.cross_entropy(prompted_clip_logits(images), dino_pseudo_labels[micro_ids].cuda(),
                                   label_smoothing=prompt_label_smoothing, reduction="sum") / len(ids)
            loss.backward()
        optimizer.step()
        updates += 1
        now = time.monotonic()
        if now-last_report >= 300:
            mins_left = (now-start)*(planned-updates)/updates/60
            print(f"Epoch {epoch}/{prompt_epochs}, update {updates}/{planned}, rough training ETA {mins_left:.0f} min", flush=True)
            last_report = now
    if recipe == "release" or epoch in (1, prompt_epochs):
        correct = prompted_clip_correct()
        print(f"Epoch {epoch}/{prompt_epochs}: {eval_split} correct={correct}/{len(eval_rows)}", flush=True)
        if recipe == "release" and correct > best_val:
            best_val, best_epoch = correct, epoch
            # Only save the trainable values; the starting CLIP weights are already on disk.
            snapshot = [p.detach().cpu().clone() for p in adapted_params]
            torch.save(dict(values=snapshot, epoch=best_epoch, val_correct=best_val), checkpoint_path)

print(f"{eval_split} out of {len(eval_rows)}: CDE={clip_correct}, DINO head={dino_head_correct}, prompted before={before}")
if recipe == "release":
    print(f"Best validation: {best_val}/{len(eval_rows)} at epoch {best_epoch}; saved {checkpoint_path}")

### Optional: score the best release checkpoint on test
Only run after locking the recipe on validation. This loads the best saved prompt, classifier, and visual normalization weights into the model initialized by §7. This is an evaluation checkpoint, not an optimizer-resume checkpoint. The teaching preset does not use this cell.

In [ ]:
if recipe == "release":
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    with torch.no_grad():
        for param, value in zip(adapted_params, saved["values"], strict=True):
            param.copy_(value)
    final_rows = split["test"]
    labels = torch.tensor([label for _, label, _ in final_rows])
    correct = prompted_clip_correct(final_rows, labels)
    print(f"NoLA test: {correct}/{len(final_rows)} ({correct/len(final_rows):.2%}); chosen at validation epoch {saved['epoch']}")

### NoLA hyperparameter reference

These are reported settings and code defaults, not necessarily optimal settings for a 160-image subset. The first cell defines all adjustable teaching settings. The default prompt count matches the released configuration; the schedule remains smaller than the paper’s.

| Setting | Authors’ released code | Teaching notebook |
| --- | --- | --- |
| Anchor cap per predicted class | `max(16, floor(0.2 × adaptation_images / classes))`; the [preprint](https://arxiv.org/html/2411.19346v2) additionally caps it at 512, which the released `find_k()` does not enforce | 4 |
| DINO head | Frozen ViT-B/16, 768 → 256 → classes, 50 epochs, batch 32, Adam default LR 0.001, ordinary cross-entropy | Same architecture and optimizer, 3 epochs, batch 16 |
| Prompted CLIP | ViT-B/32, 16 visual tokens by runtime default, 50 max epochs, batch 512, AdamW LR 0.004, label smoothing 0.1 | 16 tokens, 50 epochs in the current teaching settings, batch 8, AdamW LR 0.001, smoothing 0.1 |
| CLIP weight decay | 0.01 on regular optimizer group, 0 on the excluded normalization/bias names | 0.01 on non-bias group, 0 on biases |
| Model selection | Track best validation checkpoint | Report two preset checkpoints for comparison; do not tune against the test set |

Sources: [released config](upstream/config.yml), [runtime defaults](upstream/utils/misc.py), [training code](upstream/trainers/NoLA.py), and [arXiv v2 Tables 7–8](https://arxiv.org/html/2411.19346v2#Sx11). The sources disagree: the preprint main text says CLIP AdamW / LR 0.002, its supplement lists Adam / 0.004, and the released trainer uses AdamW / 0.004. The supplement’s 16 visual tokens match the executable default despite a trainer comment saying 50. For DINO, the supplement lists AdamW and smoothed loss; the released loop uses Adam and ordinary cross-entropy.

## After you finish EuroSAT
Compare the three classifiers on the same held-out images. A lower prompted CLIP loss need not mean more correct predictions. If a cell surprises you, stop there and inspect its output before changing a setting.

For the two follow-ups, use another reported benchmark such as Oxford Pets, then ImageNet if the dataset and compute are accessible. Port the paths, split and text inputs first; keep the NoLA cells recognizable. After those, pick a newer classification dataset and check its labels, split and description needs. If its adaptation set truly has no labels, the released selector needs a small change to take class IDs from the declared vocabulary instead of ground truth. We can make that change when we need it rather than adding generic machinery here.